# 06 — Week 2 Projections

This notebook converts the updated weekly team-strength ratings into game level projections for the selected target week.

The original preseason notebooks and preseason projections remain frozen.

For each target week matchup, this notebook:

- loads the updated weekly team strengths from Notebook 04
- preserves the original preseason Week 2 projection as a baseline
- recalculates expected margin using weekly strength, historical home field advantage, and rest
- converts expected margin into win probability using historical NFL margin variability
- carries forward the frozen preseason scoring baseline created for Week 1
- adds the new in season offensive and defensive scoring evidence
- produces projected scores, model spreads, ATS comparisons, totals comparisons, and final output tables
- saves all outputs only inside the weekly layer

The market is used only for comparison. It never enters the model projection itself.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import norm
from sklearn.linear_model import LinearRegression


## Paths and Weekly Settings

For Week 2, `TARGET_WEEK = 2`.

Future weeks use the same notebook by changing the target week after the preceding weekly notebooks have been rerun.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"
WEEKLY_OUTPUT_DIR = PROJECT_ROOT / "weekly_projections" / "outputs"

SEASON = 2026
TARGET_WEEK = 9

WEEK_OUTPUT_DIR = (
    WEEKLY_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}"
)

WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

INJURY_ADJUSTED_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

TARGET_SCHEDULE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_schedule.parquet"
)

PRESEASON_BASELINE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_preseason_baseline.parquet"
)

WEEK1_PROJECTIONS_PATH = (
    WEEKLY_DATA_DIR
    / "week_01_projections.parquet"
)

# Load Weekly Inputs

Notebook 04 supplies the updated ratings. Notebook 02 supplies the target week schedule and frozen preseason comparison.


In [3]:
weekly_strength = pd.read_parquet(WEEKLY_STRENGTH_PATH)

injury_adjusted_strength = pd.read_parquet(
    INJURY_ADJUSTED_STRENGTH_PATH
)

target_schedule = pd.read_parquet(
    TARGET_SCHEDULE_PATH
)

preseason_baseline = pd.read_parquet(
    PRESEASON_BASELINE_PATH
)

target_schedule = target_schedule[
    target_schedule["home_score"].isna()
    & target_schedule["away_score"].isna()
].copy()


preseason_baseline = preseason_baseline[
    preseason_baseline["game_id"].isin(
        target_schedule["game_id"]
    )
].copy()


print(
    "Sequential weekly strengths:",
    weekly_strength.shape
)

print(
    "Injury-adjusted strengths:",
    injury_adjusted_strength.shape
)

print(
    "Remaining target-week games:",
    target_schedule.shape
)

print(
    "Frozen preseason games:",
    preseason_baseline.shape
)

assert len(weekly_strength) == 32
assert weekly_strength["team"].nunique() == 32

assert len(injury_adjusted_strength) == 32
assert injury_adjusted_strength["team"].nunique() == 32

assert len(target_schedule) == len(
    preseason_baseline
)

required_injury_columns = [
    "team",
    "weekly_team_strength",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

missing_injury_columns = [
    col
    for col in required_injury_columns
    if col not in injury_adjusted_strength.columns
]

assert not missing_injury_columns, (
    "Missing injury-adjusted columns: "
    + ", ".join(missing_injury_columns)
)


print(
    "Using sequential ratings as the pre-injury baseline "
    "and injury-adjusted ratings for Week "
    f"{TARGET_WEEK} projections."
)

Sequential weekly strengths: (32, 25)
Injury-adjusted strengths: (32, 30)
Remaining target-week games: (15, 46)
Frozen preseason games: (15, 16)
Using sequential ratings as the pre-injury baseline and injury-adjusted ratings for Week 9 projections.


### Injury adjusted game strength

For Week 2, the `home_weekly_team_strength` and `away_weekly_team_strength` values now include the conservative injury adjustment from Notebook 05.

The injury adjustment remains visible separately in the saved team strength file, so we can distinguish:

**preseason → performance update → injury update → final Week 2 rating**


# Attach Updated Team Strength

The matchup calculation is identical in structure to the original preseason game prediction model. The only change is the source of team strength:

`preseason team_strength` → `weekly_team_strength`

This isolates the in season update without changing the original model.


In [4]:
strength_lookup = injury_adjusted_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjustment",
        "injury_adjusted_team_strength",
        "preseason_team_strength",
        "team_strength_change",
        "cumulative_offense_adjustment",
        "cumulative_defense_adjustment"
    ]
].copy()


games = target_schedule.copy()

home_lookup = strength_lookup.rename(
    columns={
        "team": "home_team",

        "weekly_team_strength":
            "home_pre_injury_team_strength",

        "injury_adjustment":
            "home_injury_adjustment",

        "injury_adjusted_team_strength":
            "home_weekly_team_strength",

        "preseason_team_strength":
            "home_preseason_team_strength",

        "team_strength_change":
            "home_strength_change",

        "cumulative_offense_adjustment":
            "home_offense_adjustment",

        "cumulative_defense_adjustment":
            "home_defense_adjustment"
    }
)

away_lookup = strength_lookup.rename(
    columns={
        "team": "away_team",

        "weekly_team_strength":
            "away_pre_injury_team_strength",

        "injury_adjustment":
            "away_injury_adjustment",

        "injury_adjusted_team_strength":
            "away_weekly_team_strength",

        "preseason_team_strength":
            "away_preseason_team_strength",

        "team_strength_change":
            "away_strength_change",

        "cumulative_offense_adjustment":
            "away_offense_adjustment",

        "cumulative_defense_adjustment":
            "away_defense_adjustment"
    }
)


games = (
    games
    .merge(
        home_lookup,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_lookup,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

games["weekly_neutral_strength_diff"] = (
    games["home_weekly_team_strength"]
    - games["away_weekly_team_strength"]
)

games["pre_injury_neutral_strength_diff"] = (
    games["home_pre_injury_team_strength"]
    - games["away_pre_injury_team_strength"]
)


games["preseason_neutral_strength_diff"] = (
    games["home_preseason_team_strength"]
    - games["away_preseason_team_strength"]
)

assert games[
    "home_weekly_team_strength"
].notna().all()

assert games[
    "away_weekly_team_strength"
].notna().all()

assert games[
    "home_injury_adjustment"
].notna().all()

assert games[
    "away_injury_adjustment"
].notna().all()

In [5]:

# Verify injury adjustments entering Week 9 projections

audit = injury_adjusted_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjustment",
        "injury_adjusted_team_strength"
    ]
].copy()

audit["calculated_strength"] = (
    audit["weekly_team_strength"]
    + audit["injury_adjustment"]
)

assert len(audit) == 32
assert audit["team"].nunique() == 32

assert np.allclose(
    audit["calculated_strength"],
    audit["injury_adjusted_team_strength"]
)

print("Projection week:", TARGET_WEEK)
print("Teams validated:", len(audit))
print(
    "Teams with active injury adjustments:",
    int((audit["injury_adjustment"] != 0).sum())
)
print(
    "Total applied injury adjustment:",
    round(audit["injury_adjustment"].sum(), 3)
)

display(
    audit.sort_values("injury_adjustment").round(3)
)

if (audit["injury_adjustment"] == 0).all():
    print(
        "WARNING: No verified injury penalties are currently "
        "applied. Projections will be preliminary."
    )


Projection week: 9
Teams validated: 32
Teams with active injury adjustments: 32
Total applied injury adjustment: -33.164


,team,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength,calculated_strength
16,BAL,-0.355,-4.500,-4.855,-4.855
24,PHI,-3.331,-3.074,-6.406,-6.406
27,ARI,-4.337,-1.877,-6.214,-6.214
12,GB,2.104,-1.630,0.474,0.474
25,LAC,-3.425,-1.337,-4.761,-4.761
18,WAS,-1.602,-1.305,-2.907,-2.907
7,DET,3.225,-1.282,1.944,1.944
2,LA,4.376,-1.202,3.174,3.174
28,CLE,-4.352,-1.193,-5.545,-5.545
26,CAR,-3.816,-1.144,-4.961,-4.961


# Home Field and Rest

These effects are recalculated using the same historical 2015–2025 methodology as the frozen preseason game prediction notebook.

They are not retuned using Week 1.


In [6]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & (pl.col("location") == "Home")
        & pl.col("result").is_not_null()
    )
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()
rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

print("Historical HFA:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest points/day:", round(REST_POINT_VALUE, 3))


Historical HFA: 1.764
Rest points/day: 0.181


In [7]:
games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"]
    - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"]
    * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["weekly_neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)


# Win Probabilities

Historical 2015–2025 game-margin variability is retained exactly as the uncertainty scale. Week 1 is not used to recalibrate probability confidence.


In [8]:
historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.50,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

print("Historical margin SD:", round(historical_margin_std, 3))


Historical margin SD: 14.201


# Preseason vs Weekly Margin Movement

The frozen preseason Week 2 projection is attached for diagnostics only.

This makes it possible to see exactly how much the Week 1 evidence changed each Week 2 matchup.


In [9]:
preseason_compare = preseason_baseline[
    [
        "game_id",
        "expected_home_margin",
        "predicted_winner",
        "predicted_win_probability"
    ]
].rename(columns={
    "expected_home_margin": "preseason_expected_home_margin",
    "predicted_winner": "preseason_predicted_winner",
    "predicted_win_probability": "preseason_predicted_win_probability"
})

games = games.merge(
    preseason_compare,
    on="game_id",
    how="left",
    validate="one_to_one"
)

games["margin_change_from_preseason"] = (
    games["expected_home_margin"]
    - games["preseason_expected_home_margin"]
)

display(
    games[
        [
            "away_team",
            "home_team",
            "preseason_expected_home_margin",
            "expected_home_margin",
            "margin_change_from_preseason",
            "preseason_predicted_winner",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ].round(3)
)


,away_team,home_team,preseason_expected_home_margin,expected_home_margin,margin_change_from_preseason,preseason_predicted_winner,predicted_winner,predicted_win_probability
0,JAX,BAL,3.125,-6.612,-9.737,BAL,JAX,0.679
1,ARI,SEA,9.624,11.292,1.668,SEA,SEA,0.787
2,CIN,ATL,-0.592,-3.446,-2.854,CIN,CIN,0.596
3,CLE,NO,5.625,5.417,-0.208,NO,NO,0.649
4,DAL,IND,2.682,0.137,-2.545,IND,IND,0.504
5,DEN,CAR,-5.796,-6.027,-0.230,DEN,DEN,0.664
6,DET,MIA,-4.698,-5.368,-0.670,DET,DET,0.647
7,GB,NE,0.594,2.547,1.953,NE,NE,0.571
8,HOU,LAC,-1.403,-4.433,-3.030,HOU,HOU,0.623
9,LA,WAS,-5.001,-4.318,0.683,LA,LA,0.619


# Frozen Preseason Scoring Ratings

Week 1's saved projection artifact contains the preseason offensive/defensive scoring ratings and personnel scoring adjustments for all 32 teams.

Those values were created before the season and are reused here as the frozen scoring prior. We do not rebuild or modify the preseason notebooks.

The new weekly offense/defense signals are added separately afterward.


In [10]:
week1_frozen = pd.read_parquet(
    WEEK1_PROJECTIONS_PATH
)

home_scoring = week1_frozen[
    [
        "home_team",
        "home_projected_off_ppg",
        "home_projected_def_ppg",
        "home_off_personnel_z",
        "home_def_personnel_z"
    ]
].rename(columns={
    "home_team": "team",
    "home_projected_off_ppg": "preseason_projected_off_ppg",
    "home_projected_def_ppg": "preseason_projected_def_ppg",
    "home_off_personnel_z": "off_personnel_z",
    "home_def_personnel_z": "def_personnel_z"
})

away_scoring = week1_frozen[
    [
        "away_team",
        "away_projected_off_ppg",
        "away_projected_def_ppg",
        "away_off_personnel_z",
        "away_def_personnel_z"
    ]
].rename(columns={
    "away_team": "team",
    "away_projected_off_ppg": "preseason_projected_off_ppg",
    "away_projected_def_ppg": "preseason_projected_def_ppg",
    "away_off_personnel_z": "off_personnel_z",
    "away_def_personnel_z": "def_personnel_z"
})

scoring_prior = (
    pd.concat([home_scoring, away_scoring], ignore_index=True)
    .groupby("team", as_index=False)
    .agg(
        preseason_projected_off_ppg=("preseason_projected_off_ppg", "mean"),
        preseason_projected_def_ppg=("preseason_projected_def_ppg", "mean"),
        off_personnel_z=("off_personnel_z", "mean"),
        def_personnel_z=("def_personnel_z", "mean")
    )
)

assert len(scoring_prior) == 32
display(scoring_prior.head())


,team,preseason_projected_off_ppg,preseason_projected_def_ppg,off_personnel_z,def_personnel_z
0,ARI,22.191473,24.425844,-0.763765,-1.345326
1,ATL,22.032425,23.261103,0.507936,0.513715
2,BAL,25.007820,22.580763,0.442967,0.969729
3,BUF,25.942602,22.289034,1.060099,0.231615
4,CAR,20.533474,23.731353,-1.506345,-0.346889


In [11]:
completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

completed_games = pd.read_parquet(
    completed_games_path
).copy()

completed_games = completed_games[
    completed_games["week"] < TARGET_WEEK
].copy()

home_scoring_actual = completed_games[
    [
        "home_team",
        "home_score",
        "away_score"
    ]
].rename(
    columns={
        "home_team": "team",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_scoring_actual = completed_games[
    [
        "away_team",
        "away_score",
        "home_score"
    ]
].rename(
    columns={
        "away_team": "team",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_scoring_actual = pd.concat(
    [
        home_scoring_actual,
        away_scoring_actual
    ],
    ignore_index=True
)

inseason_scoring = (
    team_scoring_actual
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean")
    )
)

assert len(inseason_scoring) == 32

assert (
    inseason_scoring["games_played"] > 0
).all(), "Some teams have no completed games."

assert (
    inseason_scoring["games_played"] <= TARGET_WEEK - 1
).all(), "A team has more completed games than expected."

print("Games played distribution:")
print(
    inseason_scoring["games_played"]
    .value_counts()
    .sort_index()
)

print("\nTeams with fewer completed games:")
display(
    inseason_scoring.loc[
        inseason_scoring["games_played"] < TARGET_WEEK - 1,
        ["team", "games_played"]
    ].sort_values("games_played")
)

display(
    inseason_scoring
    .sort_values("actual_off_ppg", ascending=False)
    .round(2)
)

Games played distribution:
games_played
7    14
8    18
Name: count, dtype: int64

Teams with fewer completed games:


,team,games_played
3,BUF,7
4,CAR,7
6,CIN,7
10,DET,7
12,HOU,7
14,JAX,7
15,KC,7
17,LAC,7
19,MIA,7
20,MIN,7


,team,games_played,actual_off_ppg,actual_def_ppg
3,BUF,7,33.29,25.86
6,CIN,7,29.43,22.57
10,DET,7,29.00,25.71
28,SF,7,28.14,19.86
14,JAX,7,28.00,15.86
18,LV,8,27.62,28.75
8,DAL,8,27.38,26.62
5,CHI,8,26.75,20.00
4,CAR,7,25.43,27.57
16,LA,8,24.25,21.00


In [12]:
scoring_expectations = scoring_prior[
    [
        "team",
        "preseason_projected_off_ppg",
        "preseason_projected_def_ppg"
    ]
].copy()

home_games = completed_games[
    [
        "week",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_games = home_games.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_games = completed_games[
    [
        "week",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_games = away_games.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_games = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

opponent_defense = scoring_expectations[
    [
        "team",
        "preseason_projected_def_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_def_ppg":
            "opponent_expected_def_ppg"
    }
)

opponent_offense = scoring_expectations[
    [
        "team",
        "preseason_projected_off_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_off_ppg":
            "opponent_expected_off_ppg"
    }
)

team_games = (
    team_games
    .merge(
        opponent_defense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
    .merge(
        opponent_offense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
)

league_preseason_off_ppg = (
    scoring_prior[
        "preseason_projected_off_ppg"
    ].mean()
)

league_preseason_def_ppg = (
    scoring_prior[
        "preseason_projected_def_ppg"
    ].mean()
)

team_games["adjusted_points_for"] = (
    team_games["points_for"]
    + (
        league_preseason_def_ppg
        - team_games["opponent_expected_def_ppg"]
    )
)

team_games["adjusted_points_against"] = (
    team_games["points_against"]
    + (
        league_preseason_off_ppg
        - team_games["opponent_expected_off_ppg"]
    )
)

inseason_scoring_adjusted = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean"),
        adjusted_off_ppg=(
            "adjusted_points_for",
            "mean"
        ),
        adjusted_def_ppg=(
            "adjusted_points_against",
            "mean"
        )
    )
)

assert len(inseason_scoring_adjusted) == 32

display(
    inseason_scoring_adjusted[
        [
            "team",
            "actual_off_ppg",
            "adjusted_off_ppg",
            "actual_def_ppg",
            "adjusted_def_ppg"
        ]
    ]
    .sort_values(
        "adjusted_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,actual_off_ppg,adjusted_off_ppg,actual_def_ppg,adjusted_def_ppg
3,BUF,33.29,33.78,25.86,25.24
6,CIN,29.43,29.53,22.57,22.56
10,DET,29.00,28.85,25.71,26.36
14,JAX,28.00,28.34,15.86,15.71
28,SF,28.14,28.11,19.86,19.60
18,LV,27.62,27.98,28.75,28.77
8,DAL,27.38,27.37,26.62,26.43
5,CHI,26.75,27.06,20.00,20.42
4,CAR,25.43,25.58,27.57,27.35
16,LA,24.25,24.43,21.00,21.10


In [13]:
scoring_model = scoring_prior.merge(
    inseason_scoring_adjusted,
    on="team",
    how="left",
    validate="one_to_one"
)

PRESEASON_EQUIVALENT_GAMES = 5.0

scoring_model["preseason_weight"] = (
    PRESEASON_EQUIVALENT_GAMES
    / (
        PRESEASON_EQUIVALENT_GAMES
        + scoring_model["games_played"]
    )
)

scoring_model["inseason_weight"] = (
    1.0
    - scoring_model["preseason_weight"]
)

scoring_model["current_off_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_off_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_off_ppg"]
)

scoring_model["current_def_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_def_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_def_ppg"]
)

assert len(scoring_model) == 32

assert scoring_model[
    [
        "current_off_ppg",
        "current_def_ppg"
    ]
].notna().all().all()

display(
    scoring_model[
        [
            "team",
            "preseason_projected_off_ppg",
            "adjusted_off_ppg",
            "current_off_ppg",
            "preseason_projected_def_ppg",
            "adjusted_def_ppg",
            "current_def_ppg",
            "preseason_weight",
            "inseason_weight"
        ]
    ]
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,preseason_projected_off_ppg,adjusted_off_ppg,current_off_ppg,preseason_projected_def_ppg,adjusted_def_ppg,current_def_ppg,preseason_weight,inseason_weight
3,BUF,25.94,33.78,30.51,22.29,25.24,24.01,0.42,0.58
10,DET,26.28,28.85,27.78,23.05,26.36,24.98,0.42,0.58
6,CIN,23.96,29.53,27.21,24.55,22.56,23.39,0.42,0.58
28,SF,24.31,28.11,26.53,22.70,19.60,20.89,0.42,0.58
14,JAX,23.95,28.34,26.51,22.47,15.71,18.53,0.42,0.58
8,DAL,24.71,27.37,26.35,24.75,26.43,25.79,0.38,0.62
5,CHI,23.20,27.06,25.58,23.18,20.42,21.48,0.38,0.62
16,LA,25.21,24.43,24.73,22.35,21.10,21.58,0.38,0.62
18,LV,19.47,27.98,24.71,23.59,28.77,26.78,0.38,0.62
27,SEA,24.46,23.36,23.78,21.63,19.80,20.51,0.38,0.62


# Weekly Scoring Update

The in-season offensive and defensive adjustments from Notebook 04 are already shrunk for sample size.

They are added to the frozen scoring prior:

- positive offensive adjustment raises expected scoring
- positive defensive adjustment represents points saved, so it lowers projected points allowed

This keeps the Week 1 scoring evidence separate and interpretable.


In [14]:
scoring_update = scoring_model[
    [
        "team",
        "current_off_ppg",
        "current_def_ppg"
    ]
].copy()

assert len(scoring_update) == 32
assert scoring_update["team"].nunique() == 32

display(
    scoring_update
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,current_off_ppg,current_def_ppg
3,BUF,30.51,24.01
10,DET,27.78,24.98
6,CIN,27.21,23.39
28,SF,26.53,20.89
14,JAX,26.51,18.53
8,DAL,26.35,25.79
5,CHI,25.58,21.48
16,LA,24.73,21.58
18,LV,24.71,26.78
27,SEA,23.78,20.51


# Weekly Projected Total

The matchup total uses the same basic preseason scoring concept: each offense is paired with the opposing defense.

Because the weekly offense/defense inputs are already heavily shrunk, no additional Week 1 multiplier is applied here.

The final projected scores are forced to satisfy both:

- projected score difference = weekly expected margin
- projected score sum = weekly projected total


In [15]:
home_scoring_update = scoring_update.rename(
    columns={
        "team": "home_team",
        "current_off_ppg":
            "home_current_off_ppg",
        "current_def_ppg":
            "home_current_def_ppg"
    }
)

away_scoring_update = scoring_update.rename(
    columns={
        "team": "away_team",
        "current_off_ppg":
            "away_current_off_ppg",
        "current_def_ppg":
            "away_current_def_ppg"
    }
)

In [16]:
games = (
    games
    .merge(
        home_scoring_update,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring_update,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

required_scoring_columns = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[
    required_scoring_columns
].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2


assert games["projected_total"].notna().all()

assert np.allclose(
    games["projected_home_score"]
    - games["projected_away_score"],
    games["expected_home_margin"]
)

assert np.allclose(
    games["projected_home_score"]
    + games["projected_away_score"],
    games["projected_total"]
)


display(
    games[
        [
            "away_team",
            "home_team",
            "away_current_off_ppg",
            "home_current_def_ppg",
            "home_current_off_ppg",
            "away_current_def_ppg",
            "projected_away_score",
            "projected_home_score",
            "projected_total",
            "expected_home_margin"
        ]
    ].round(2)
)

,away_team,home_team,away_current_off_ppg,home_current_def_ppg,home_current_off_ppg,away_current_def_ppg,projected_away_score,projected_home_score,projected_total,expected_home_margin
0,JAX,BAL,26.51,24.40,23.00,18.53,26.41,19.80,46.22,-6.61
1,ARI,SEA,22.13,20.51,23.78,27.04,17.72,29.01,46.73,11.29
2,CIN,ATL,27.21,23.46,21.70,23.39,25.66,22.21,47.87,-3.45
3,CLE,NO,18.24,25.76,22.32,22.51,19.50,24.92,44.41,5.42
4,DAL,IND,26.35,26.23,21.92,25.79,25.00,25.14,50.14,0.14
5,DEN,CAR,22.59,25.84,23.48,20.71,26.17,20.14,46.31,-6.03
6,DET,MIA,27.78,25.16,19.06,24.98,26.93,21.56,48.49,-5.37
7,GB,NE,21.92,21.06,23.15,21.19,20.56,23.10,43.66,2.55
8,HOU,LAC,22.92,23.50,17.89,22.82,24.00,19.57,43.57,-4.43
9,LA,WAS,24.73,24.78,22.81,21.58,25.64,21.32,46.95,-4.32


# Market Comparison

nflverse uses positive `spread_line` for a home favorite and negative for an away favorite. The values below are converted to standard sportsbook notation.

Market information is used only after the model projections have been completed.


In [17]:
games["home_spread"] = -games["spread_line"]
games["away_spread"] = games["spread_line"]

games["model_home_spread"] = -games["expected_home_margin"]
games["model_away_spread"] = games["expected_home_margin"]

games["home_ats_edge"] = (
    games["home_spread"]
    - games["model_home_spread"]
)

games["away_ats_edge"] = (
    games["away_spread"]
    - games["model_away_spread"]
)

games["ats_pick"] = np.where(
    games["home_ats_edge"] > games["away_ats_edge"],
    games["home_team"],
    np.where(
        games["away_ats_edge"] > games["home_ats_edge"],
        games["away_team"],
        "PUSH"
    )
)

games["ats_difference"] = (
    games[["home_ats_edge", "away_ats_edge"]]
    .max(axis=1)
)

games["total_difference_signed"] = (
    games["projected_total"]
    - games["total_line"]
)

games["total_difference"] = (
    games["total_difference_signed"].abs()
)

games["total_pick"] = np.where(
    games["total_difference_signed"] > 0,
    "OVER",
    np.where(
        games["total_difference_signed"] < 0,
        "UNDER",
        "PUSH"
    )
)


## Unit Sizing

Unit sizes are assigned from the model's edge over the market.

ATS and totals use separate thresholds because the totals model is newer and requires a larger edge before qualifying as a play.

These units do not change any projections or picks. They provide a consistent way to track performance by model confidence.

In [18]:
def assign_ats_units(edge):
    if pd.isna(edge) or edge < 1.0:
        return 0.0
    elif edge < 2.0:
        return 0.5
    elif edge < 3.0:
        return 1.0
    elif edge < 4.0:
        return 1.5
    else:
        return 2.0


def assign_total_units(edge):
    if pd.isna(edge) or edge < 2.0:
        return 0.0
    elif edge < 3.5:
        return 0.5
    elif edge < 5.0:
        return 1.0
    elif edge < 7.0:
        return 1.5
    else:
        return 2.0


games["ats_units"] = games["ats_difference"].apply(assign_ats_units)

games["total_units"] = games["total_difference"].apply(assign_total_units)


games[
    [
        "away_team",
        "home_team",
        "ats_pick",
        "ats_difference",
        "ats_units",
        "total_pick",
        "total_difference",
        "total_units"
    ]
]

,away_team,home_team,ats_pick,ats_difference,ats_units,total_pick,total_difference,total_units
0,JAX,BAL,BAL,3.387753,1.5,OVER,6.715092,1.5
1,ARI,SEA,SEA,2.792080,1.0,OVER,7.227016,2.0
2,CIN,ATL,CIN,0.446006,0.0,UNDER,4.625671,1.0
3,CLE,NO,NO,2.416903,1.0,OVER,1.913300,0.0
4,DAL,IND,IND,5.637091,2.0,UNDER,1.357290,0.0
5,DEN,CAR,DEN,3.526633,1.5,OVER,1.810195,0.0
6,DET,MIA,MIA,1.631650,0.5,UNDER,2.013032,0.5
7,GB,NE,GB,0.452629,0.0,UNDER,1.840739,0.0
8,HOU,LAC,LAC,1.066966,0.5,OVER,3.070894,0.5
9,LA,WAS,WAS,0.182360,0.0,UNDER,1.547284,0.0


# Display Helpers


In [19]:
def format_spread(team, line):
    if pd.isna(line):
        return "N/A"
    if line > 0:
        return f"{team} +{line:.1f}"
    return f"{team} {line:.1f}"

def model_spread_display(row):
    if row["model_home_spread"] < 0:
        return format_spread(row["home_team"], row["model_home_spread"])
    if row["model_away_spread"] < 0:
        return format_spread(row["away_team"], row["model_away_spread"])
    return "PICK"

def market_spread_display(row):
    if pd.isna(row["spread_line"]):
        return "N/A"
    if row["home_spread"] < 0:
        return format_spread(row["home_team"], row["home_spread"])
    if row["away_spread"] < 0:
        return format_spread(row["away_team"], row["away_spread"])
    return "PICK"

def ats_pick_display(row):
    if row["ats_pick"] == "PUSH":
        return "PUSH"
    if row["ats_pick"] == row["home_team"]:
        return format_spread(row["home_team"], row["home_spread"])
    return format_spread(row["away_team"], row["away_spread"])

games["projected_score"] = (
    games["away_team"]
    + " "
    + games["projected_away_score"].round(1).astype(str)
    + " - "
    + games["home_team"]
    + " "
    + games["projected_home_score"].round(1).astype(str)
)

games["model_spread_display"] = games.apply(
    model_spread_display, axis=1
)

games["market_spread_display"] = games.apply(
    market_spread_display, axis=1
)

games["ats_pick_display"] = games.apply(
    ats_pick_display, axis=1
)


# Final Week Projection Board


In [20]:

projection_board = pd.DataFrame({
    "Matchup": games["away_team"] + " @ " + games["home_team"],
    "Projected Score": games["projected_score"],
    "SU Pick": games["predicted_winner"],
    "Win Prob": games["predicted_win_probability"],
    "Model Spread": games["model_spread_display"],
    "Market Spread": games["market_spread_display"],
    "ATS Pick": games["ats_pick_display"],
    "ATS Diff": games["ats_difference"],
    "ATS Units": games["ats_units"].map(lambda x: f"{x:.1f}"),
    "Model Total": games["projected_total"],
    "Market Total": games["total_line"],
    "O/U Pick": games["total_pick"],
    "Total Diff": games["total_difference"],
    "Total Units": games["total_units"].map(lambda x: f"{x:.1f}"),
    "Preseason Margin": games["preseason_expected_home_margin"],
    "Weekly Margin": games["expected_home_margin"],
    "Margin Change": games["margin_change_from_preseason"]
})

projection_display = projection_board.copy()

projection_csv_path = (
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_projections.csv"
)

if projection_csv_path.exists():
    previous_board = pd.read_csv(projection_csv_path)

    current_matchups = set(projection_display["Matchup"])

    frozen_board = previous_board.loc[
        ~previous_board["Matchup"].isin(current_matchups)
    ].copy()

    projection_display = pd.concat(
        [frozen_board, projection_display],
        ignore_index=True
    )

assert projection_display["Matchup"].is_unique, (
    "Duplicate matchups found in projection board."
)

display_board = projection_display.copy()

numeric_columns = [
    "Win Prob",
    "ATS Diff",
    "ATS Units",
    "Model Total",
    "Market Total",
    "Total Diff",
    "Total Units",
    "Preseason Margin",
    "Weekly Margin",
    "Margin Change"
]

for col in numeric_columns:
    display_board[col] = pd.to_numeric(
        display_board[col],
        errors="raise"
    )

print(
    f"Week {TARGET_WEEK} Projection Board "
    f"({len(display_board)} games)"
)

display(
    display_board.style.format({
        "Win Prob": "{:.3f}",
        "ATS Diff": "{:.2f}",
        "ATS Units": "{:.1f}",
        "Model Total": "{:.1f}",
        "Market Total": "{:.1f}",
        "Total Diff": "{:.1f}",
        "Total Units": "{:.1f}",
        "Preseason Margin": "{:.2f}",
        "Weekly Margin": "{:.2f}",
        "Margin Change": "{:+.2f}"
    }).hide(axis="index")
)


Week 9 Projection Board (15 games)


Matchup,Projected Score,SU Pick,Win Prob,Model Spread,Market Spread,ATS Pick,ATS Diff,ATS Units,Model Total,Market Total,O/U Pick,Total Diff,Total Units,Preseason Margin,Weekly Margin,Margin Change
JAX @ BAL,JAX 26.4 - BAL 19.8,JAX,0.679,JAX -6.6,JAX -10.0,BAL +10.0,3.39,1.5,46.2,39.5,OVER,6.7,1.5,3.12,-6.61,-9.74
ARI @ SEA,ARI 17.7 - SEA 29.0,SEA,0.787,SEA -11.3,SEA -8.5,SEA -8.5,2.79,1.0,46.7,39.5,OVER,7.2,2.0,9.62,11.29,+1.67
CIN @ ATL,CIN 25.7 - ATL 22.2,CIN,0.596,CIN -3.4,CIN -3.0,CIN -3.0,0.45,0.0,47.9,52.5,UNDER,4.6,1.0,-0.59,-3.45,-2.85
CLE @ NO,CLE 19.5 - NO 24.9,NO,0.649,NO -5.4,NO -3.0,NO -3.0,2.42,1.0,44.4,42.5,OVER,1.9,0.0,5.62,5.42,-0.21
DAL @ IND,DAL 25.0 - IND 25.1,IND,0.504,IND -0.1,DAL -5.5,IND +5.5,5.64,2.0,50.1,51.5,UNDER,1.4,0.0,2.68,0.14,-2.55
DEN @ CAR,DEN 26.2 - CAR 20.1,DEN,0.664,DEN -6.0,DEN -2.5,DEN -2.5,3.53,1.5,46.3,44.5,OVER,1.8,0.0,-5.80,-6.03,-0.23
DET @ MIA,DET 26.9 - MIA 21.6,DET,0.647,DET -5.4,DET -7.0,MIA +7.0,1.63,0.5,48.5,50.5,UNDER,2.0,0.5,-4.70,-5.37,-0.67
GB @ NE,GB 20.6 - NE 23.1,NE,0.571,NE -2.5,NE -3.0,GB +3.0,0.45,0.0,43.7,45.5,UNDER,1.8,0.0,0.59,2.55,+1.95
HOU @ LAC,HOU 24.0 - LAC 19.6,HOU,0.623,HOU -4.4,HOU -5.5,LAC +5.5,1.07,0.5,43.6,40.5,OVER,3.1,0.5,-1.40,-4.43,-3.03
LA @ WAS,LA 25.6 - WAS 21.3,LA,0.619,LA -4.3,LA -4.5,WAS +4.5,0.18,0.0,47.0,48.5,UNDER,1.5,0.0,-5.00,-4.32,+0.68


# Biggest Model vs Market Differences


In [21]:
biggest_ats_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Spread",
            "Market Spread",
            "ATS Pick",
            "ATS Diff"
        ]
    ]
    .sort_values("ATS Diff", ascending=False)
    .head(5)
    .copy()
)

biggest_total_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Total",
            "Market Total",
            "O/U Pick",
            "Total Diff"
        ]
    ]
    .sort_values("Total Diff", ascending=False)
    .head(5)
    .copy()
)

print("BIGGEST ATS DIFFERENCES")
display(biggest_ats_differences.round(2))

print()
print("BIGGEST TOTAL DIFFERENCES")
display(biggest_total_differences.round(2))


BIGGEST ATS DIFFERENCES


,Matchup,Projected Score,Model Spread,Market Spread,ATS Pick,ATS Diff
4,DAL @ IND,DAL 25.0 - IND 25.1,IND -0.1,DAL -5.5,IND +5.5,5.64
5,DEN @ CAR,DEN 26.2 - CAR 20.1,DEN -6.0,DEN -2.5,DEN -2.5,3.53
0,JAX @ BAL,JAX 26.4 - BAL 19.8,JAX -6.6,JAX -10.0,BAL +10.0,3.39
1,ARI @ SEA,ARI 17.7 - SEA 29.0,SEA -11.3,SEA -8.5,SEA -8.5,2.79
13,TB @ CHI,TB 20.8 - CHI 24.4,CHI -3.5,CHI -6.0,TB +6.0,2.46



BIGGEST TOTAL DIFFERENCES


,Matchup,Projected Score,Model Total,Market Total,O/U Pick,Total Diff
1,ARI @ SEA,ARI 17.7 - SEA 29.0,46.73,39.5,OVER,7.23
0,JAX @ BAL,JAX 26.4 - BAL 19.8,46.22,39.5,OVER,6.72
11,NYG @ PHI,NYG 23.4 - PHI 20.9,44.36,38.5,OVER,5.86
2,CIN @ ATL,CIN 25.7 - ATL 22.2,47.87,52.5,UNDER,4.63
12,NYJ @ KC,NYJ 17.2 - KC 27.1,44.29,47.5,UNDER,3.21


In [22]:
biggest_ats_differences["ATS Diff"] = (
    biggest_ats_differences["ATS Diff"].round(2)
)

biggest_total_differences["Model Total"] = (
    biggest_total_differences["Model Total"].round(1)
)

biggest_total_differences["Market Total"] = (
    biggest_total_differences["Market Total"].round(1)
)

biggest_total_differences["Total Diff"] = (
    biggest_total_differences["Total Diff"].round(1)
)

# Biggest Changes from the Preseason Projection

This is one of the most important weekly diagnostics. It shows which games changed most because of new in season information.


In [23]:
biggest_projection_changes = (
    projection_board[
        [
            "Matchup",
            "Preseason Margin",
            "Weekly Margin",
            "Margin Change",
            "SU Pick",
            "Win Prob"
        ]
    ]
    .assign(
        Absolute_Change=lambda x: x["Margin Change"].abs()
    )
    .sort_values("Absolute_Change", ascending=False)
    .drop(columns="Absolute_Change")
)

display(biggest_projection_changes.round(3))


,Matchup,Preseason Margin,Weekly Margin,Margin Change,SU Pick,Win Prob
0,JAX @ BAL,3.125,-6.612,-9.737,JAX,0.679
11,NYG @ PHI,6.404,-2.468,-8.872,NYG,0.569
8,HOU @ LAC,-1.403,-4.433,-3.030,HOU,0.623
2,CIN @ ATL,-0.592,-3.446,-2.854,CIN,0.596
13,TB @ CHI,0.930,3.538,2.609,CHI,0.598
4,DAL @ IND,2.682,0.137,-2.545,IND,0.504
7,GB @ NE,0.594,2.547,1.953,NE,0.571
1,ARI @ SEA,9.624,11.292,1.668,SEA,0.787
10,LV @ SF,11.091,9.711,-1.379,SF,0.753
12,NYJ @ KC,8.999,9.877,0.877,KC,0.757


In [24]:
biggest_projection_changes["Preseason Margin"] = (
    biggest_projection_changes["Preseason Margin"].round(2)
)

biggest_projection_changes["Weekly Margin"] = (
    biggest_projection_changes["Weekly Margin"].round(2)
)

biggest_projection_changes["Margin Change"] = (
    biggest_projection_changes["Margin Change"].round(2)
)

biggest_projection_changes["Win Prob"] = (
    biggest_projection_changes["Win Prob"].round(3)
)

# Sanity Checks


In [25]:
assert len(games) == len(target_schedule)
assert games["weekly_neutral_strength_diff"].notna().all()
assert games["expected_home_margin"].notna().all()
assert games["predicted_win_probability"].between(0.5, 1.0).all()
assert games["projected_total"].notna().all()

score_margin_check = (
    games["projected_home_score"]
    - games["projected_away_score"]
)

score_total_check = (
    games["projected_home_score"]
    + games["projected_away_score"]
)

assert np.allclose(
    score_margin_check,
    games["expected_home_margin"]
)

assert np.allclose(
    score_total_check,
    games["projected_total"]
)

print("All weekly game prediction sanity checks passed.")


All weekly game prediction sanity checks passed.


# Save Week Outputs

Everything is written only to the weekly layer.

The detailed parquet preserves all modeling fields for later evaluation. The CSV files provide the clean weekly projection board and largest model versus market differences.


In [26]:

detailed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_projections.parquet"
)

current_games = games.copy()

if detailed_path.exists():
    previous_games = pd.read_parquet(detailed_path)

    current_game_ids = set(current_games["game_id"])

    frozen_games = previous_games[
        ~previous_games["game_id"].isin(current_game_ids)
    ].copy()

    detailed_save = pd.concat(
        [frozen_games, current_games],
        ignore_index=True
    )
else:
    detailed_save = current_games.copy()

detailed_save.to_parquet(
    detailed_path,
    index=False
)

# Build the complete projection board
projection_save = projection_board.copy()

projection_save["ATS Units"] = games["ats_units"].values
projection_save["Total Units"] = games["total_units"].values

rounding = {
    "Win Prob": 3,
    "ATS Diff": 2,
    "ATS Units": 1,
    "Model Total": 1,
    "Market Total": 1,
    "Total Diff": 1,
    "Total Units": 1,
    "Preseason Margin": 2,
    "Weekly Margin": 2,
    "Margin Change": 2
}

for column, decimals in rounding.items():
    projection_save[column] = projection_save[column].round(decimals)

projection_csv_path = (
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_projections.csv"
)

# Preserve previously saved games when refreshing projections
if projection_csv_path.exists():
    previous_projections = pd.read_csv(projection_csv_path)

    current_matchups = set(projection_save["Matchup"])

    frozen_projections = previous_projections[
        ~previous_projections["Matchup"].isin(current_matchups)
    ].copy()

    projection_save = pd.concat(
        [frozen_projections, projection_save],
        ignore_index=True
    )

assert projection_save["Matchup"].is_unique, (
    "Duplicate matchups found in saved projections."
)

projection_save.to_csv(
    projection_csv_path,
    index=False
)

# Save model-versus-market analysis
biggest_ats_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv",
    index=False
)

biggest_total_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv",
    index=False
)

biggest_projection_changes.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv",
    index=False
)

print("Saved:")
print(detailed_path)
print(projection_csv_path)

print(
    f"\nTotal saved Week {TARGET_WEEK} projections: "
    f"{len(projection_save)}"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv"
)


Saved:
..\..\data\processed\weekly\week_09_projections.parquet
..\..\weekly_projections\outputs\week_09\week_09_projections.csv

Total saved Week 9 projections: 15
..\..\weekly_projections\outputs\week_09\week_09_biggest_ats_differences.csv
..\..\weekly_projections\outputs\week_09\week_09_biggest_total_differences.csv
..\..\weekly_projections\outputs\week_09\week_09_biggest_projection_changes.csv


# Weekly Pipeline

For each future week, the intended order is:

`02_Weekly_Data_Update`  
→ `03_Inseason_Feature_Engineering`  
→ `04_Weekly_Team_Strength`  
→ `06_Week2_Projections`

The original preseason notebooks remain frozen throughout the season.

The next notebook will be `06_Weekly_Evaluation.ipynb`, which will compare locked weekly predictions against actual results after each week is complete.


In [27]:
ros_scoring_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_ros_scoring_ratings.parquet"
)

scoring_model[
    ["team", "current_off_ppg", "current_def_ppg"]
].to_parquet(
    ros_scoring_path,
    index=False
)

print("Saved:", ros_scoring_path)

Saved: ..\..\data\processed\weekly\week_09_ros_scoring_ratings.parquet
